<a href="https://colab.research.google.com/github/codylewis1286-spec/AI-5615-Homework/blob/main/mri_tumor_seg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Cell 1-Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Cell 2- Verify Root Path

In [ ]:
import os

root = "/content/drive/MyDrive/ML Medical Imaging/Project/Task01_BrainTumour" # adjust if needed
print("Exists:", os.path.exists(root))
if os.path.exists(root):
    print(os.listdir(root)[:10])

Cell 3- BraTSSliceDataset class

In [ ]:
import os
from pathlib import Path
from typing import List, Tuple, Optional

import nibabel as nib
import numpy as np
import torch
from torch.utils.data import Dataset


class BraTSSliceDataset(Dataset):
    """
    2D axial slice dataset for Medical Segmentation Decathlon Task01_BrainTumour.

    Each image file is expected to be a 4D NIfTI volume with shape:
        (H, W, Z, C)
    where C=4 modalities: FLAIR, T1w, T1gd, T2w

    Each label file is expected to be a 3D NIfTI volume with shape:
        (H, W, Z)

    Returns:
        image_tensor: torch.FloatTensor of shape [4, H, W]
        mask_tensor:  torch.LongTensor of shape [H, W]
    """

    def __init__(
        self,
        root_dir: str,
        split: str = "train",
        case_ids: Optional[List[str]] = None,
        tumor_only: bool = True,
        min_tumor_pixels: int = 1,
        normalize: bool = True,
        background_keep_ratio: float = 0.2,
    ):
        """
        Args:
            root_dir: Path to Task01_BrainTumour directory
            split: Currently only supports "train" because labels are needed
            case_ids: Optional list of case IDs like ["BRATS_001", "BRATS_002"]
            tumor_only: If True, keep only slices with tumor in the mask
            min_tumor_pixels: Minimum number of nonzero mask pixels for a slice to be kept
            normalize: If True, z-score normalize each modality per volume over nonzero voxels
        """
        self.root_dir = Path(root_dir)
        self.images_dir = self.root_dir / "imagesTr"
        self.labels_dir = self.root_dir / "labelsTr"

        if split != "train":
            raise ValueError("This dataset currently supports only split='train' because labels are required.")

        self.tumor_only = tumor_only
        self.min_tumor_pixels = min_tumor_pixels
        self.normalize = normalize
        self.background_keep_ratio = background_keep_ratio

        self.case_paths = self._get_case_paths(case_ids)
        self.slice_index = self._build_slice_index()

    def _get_case_paths(self, case_ids: Optional[List[str]]) -> List[Tuple[Path, Path]]:
        image_files = sorted(self.images_dir.glob("*.nii.gz"))

        if case_ids is not None:
            case_ids = set(case_ids)
            image_files = [p for p in image_files if p.stem.replace(".nii", "") in case_ids]

        case_paths = []
        for img_path in image_files:
            case_id = img_path.name.replace(".nii.gz", "")
            lab_path = self.labels_dir / f"{case_id}.nii.gz"

            if not lab_path.exists():
                print(f"Warning: missing label for {case_id}, skipping.")
                continue

            case_paths.append((img_path, lab_path))

        if len(case_paths) == 0:
            raise ValueError("No valid image/label pairs found.")

        return case_paths

    def _build_slice_index(self) -> List[Tuple[Path, Path, int]]:
        """
        Build a list of (image_path, label_path, slice_idx) entries.
        """
        slice_index = []

        for img_path, lab_path in self.case_paths:
            label = nib.load(str(lab_path)).get_fdata()

            if label.ndim != 3:
                raise ValueError(f"Expected 3D label volume for {lab_path}, got shape {label.shape}")

            num_slices = label.shape[2]

            for z in range(num_slices):
                mask_slice = label[:, :, z]
                tumor_pixels = np.sum(mask_slice > 0)

                if self.tumor_only:
                    if tumor_pixels >= self.min_tumor_pixels:
                        slice_index.append((img_path, lab_path, z))
                else:
                    slice_index.append((img_path, lab_path, z))

        if len(slice_index) == 0:
            raise ValueError("No slices found. Try setting tumor_only=False or lowering min_tumor_pixels.")

        return slice_index

    @staticmethod
    def _normalize_volume(volume: np.ndarray) -> np.ndarray:
        """
        Z-score normalize each modality independently over nonzero voxels.

        Args:
            volume: numpy array of shape (H, W, Z, C)

        Returns:
            normalized volume with same shape
        """
        volume = volume.astype(np.float32)
        normalized = np.zeros_like(volume, dtype=np.float32)

        for c in range(volume.shape[3]):
            channel = volume[:, :, :, c]
            mask = channel != 0

            if np.any(mask):
                mean = channel[mask].mean()
                std = channel[mask].std()

                if std < 1e-8:
                    std = 1.0

                normalized[:, :, :, c] = channel
                normalized[:, :, :, c][mask] = (channel[mask] - mean) / std
            else:
                normalized[:, :, :, c] = channel

        return normalized

    def __len__(self) -> int:
        return len(self.slice_index)

    def __getitem__(self, idx: int):
        img_path, lab_path, z = self.slice_index[idx]

        image = nib.load(str(img_path)).get_fdata()
        label = nib.load(str(lab_path)).get_fdata()

        if image.ndim != 4:
            raise ValueError(f"Expected 4D image volume for {img_path}, got shape {image.shape}")
        if label.ndim != 3:
            raise ValueError(f"Expected 3D label volume for {lab_path}, got shape {label.shape}")

        if self.normalize:
            image = self._normalize_volume(image)

        image_slice = image[:, :, z, :]          # (H, W, 4)
        label_slice = label[:, :, z]             # (H, W)

        # Convert image to channel-first: [4, H, W]
        image_slice = np.transpose(image_slice, (2, 0, 1)).astype(np.float32)

        # Labels should be integer class IDs
        label_slice = label_slice.astype(np.int64)

        image_tensor = torch.from_numpy(image_slice)
        mask_tensor = torch.from_numpy(label_slice)

        return image_tensor, mask_tensor


if __name__ == "__main__":
    root = "/content/drive/MyDrive/ML Medical Imaging/Project/Task01_BrainTumour"

    dataset = BraTSSliceDataset(
        root_dir=root,
        tumor_only=True,
        min_tumor_pixels=10,
        normalize=True,
    )

    print(f"Number of slices: {len(dataset)}")

    x, y = dataset[0]
    print("Image tensor shape:", x.shape)   # expected [4, H, W]
    print("Mask tensor shape:", y.shape)    # expected [H, W]
    print("Image dtype:", x.dtype)
    print("Mask dtype:", y.dtype)
    print("Unique mask labels:", torch.unique(y))

Cell 4- Split/raw dataset

In [ ]:
from collections import Counter
import torch

label_counter = Counter()

for i in range(200):  # sample first 200 slices
    _, mask = dataset[i]
    unique_vals = torch.unique(mask).tolist()
    for val in unique_vals:
        label_counter[int(val)] += 1

print(label_counter)

Cell 5- Cache-Builder

In [ ]:
from pathlib import Path
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
import os
import json

root = "/content/drive/MyDrive/ML Medical Imaging/Project/Task01_BrainTumour"
cache_root = "/content/drive/MyDrive/ML in Medical Imaging/ML Medical Imaging/Project/brats_cache"

os.makedirs(cache_root, exist_ok=True)

images_dir = Path(root) / "imagesTr"

# Get case IDs from filenames
case_ids = sorted([p.name.replace(".nii.gz", "") for p in images_dir.glob("*.nii.gz")])

print("Total cases:", len(case_ids))
print("First 5 case IDs:", case_ids[:5])

# Split by case, not by slice
train_case_ids, val_case_ids = train_test_split(
    case_ids,
    test_size=0.2,
    random_state=42
)

print("Train cases:", len(train_case_ids))
print("Val cases:", len(val_case_ids))

# Save split so future runs use the same cases
split_info = {
    "train_case_ids": train_case_ids,
    "val_case_ids": val_case_ids
}

with open(os.path.join(cache_root, "split.json"), "w") as f:
    json.dump(split_info, f, indent=2)

# Build raw datasets from NIfTI files
train_dataset_raw = BraTSSliceDataset(
    root_dir=root,
    case_ids=train_case_ids,
    tumor_only=False,
    min_tumor_pixels=10,
    normalize=True,
)

val_dataset_raw = BraTSSliceDataset(
    root_dir=root,
    case_ids=val_case_ids,
    tumor_only=False,
    min_tumor_pixels=10,
    normalize=True,
)

print("Raw train slices:", len(train_dataset_raw))
print("Raw val slices:", len(val_dataset_raw))

Cell 6- Cached Data

In [ ]:
import os
import numpy as np
import torch
from tqdm import tqdm
import json

cache_root = "/content/drive/MyDrive/ML in Medical Imaging/ML Medical Imaging/Project/brats_cache"

train_img_dir = os.path.join(cache_root, "train_images")
train_mask_dir = os.path.join(cache_root, "train_masks")
val_img_dir = os.path.join(cache_root, "val_images")
val_mask_dir = os.path.join(cache_root, "val_masks")

os.makedirs(train_img_dir, exist_ok=True)
os.makedirs(train_mask_dir, exist_ok=True)
os.makedirs(val_img_dir, exist_ok=True)
os.makedirs(val_mask_dir, exist_ok=True)

def save_dataset_to_cache(dataset, image_dir, mask_dir, prefix):
    metadata = []

    for i in tqdm(range(len(dataset))):
        image, mask = dataset[i]

        if torch.is_tensor(image):
            image = image.numpy()
        if torch.is_tensor(mask):
            mask = mask.numpy()

        image = image.astype(np.float32)   # [4, H, W]
        mask = mask.astype(np.uint8)       # [H, W]

        image_name = f"{prefix}_img_{i:05d}.npy"
        mask_name = f"{prefix}_mask_{i:05d}.npy"

        np.save(os.path.join(image_dir, image_name), image)
        np.save(os.path.join(mask_dir, mask_name), mask)

        metadata.append({
            "index": i,
            "image_file": image_name,
            "mask_file": mask_name
        })

    return metadata

train_meta = save_dataset_to_cache(train_dataset_raw, train_img_dir, train_mask_dir, "train")
val_meta = save_dataset_to_cache(val_dataset_raw, val_img_dir, val_mask_dir, "val")

with open(os.path.join(cache_root, "train_meta.json"), "w") as f:
    json.dump(train_meta, f, indent=2)

with open(os.path.join(cache_root, "val_meta.json"), "w") as f:
    json.dump(val_meta, f, indent=2)

print("Caching complete.")
print("Train cached slices:", len(train_meta))
print("Val cached slices:", len(val_meta))

Cell 7- Cached DataLoader

In [ ]:
from torch.utils.data import DataLoader
import os

cache_root = "/content/drive/MyDrive/ML in Medical Imaging/ML Medical Imaging/Project/brats_cache"

train_dataset = CachedSliceDataset(
    image_dir=os.path.join(cache_root, "train_images"),
    mask_dir=os.path.join(cache_root, "train_masks"),
)

val_dataset = CachedSliceDataset(
    image_dir=os.path.join(cache_root, "val_images"),
    mask_dir=os.path.join(cache_root, "val_masks"),
)

print("Cached train slices:", len(train_dataset))
print("Cached val slices:", len(val_dataset))

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

images, masks = next(iter(train_loader))
print("Batch image shape:", images.shape)
print("Batch mask shape:", masks.shape)
print("Labels in batch:", torch.unique(masks))

Cell 8- Model

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Example: if your model class is already defined above
model = UNet2D(in_channels=4, out_channels=4)

# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# test one batch through the model
images, masks = next(iter(train_loader))
images = images.to(device)
masks = masks.to(device)

outputs = model(images)

print("Input shape:", images.shape)     # expected: [B, 4, 240, 240]
print("Mask shape:", masks.shape)       # expected: [B, 240, 240]
print("Output shape:", outputs.shape)   # expected: [B, 4, 240, 240]

loss = criterion(outputs, masks)
print("Test loss:", loss.item())

Cell 9-

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

In [ ]:
num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, masks in train_loader:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_train_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}] Train Loss: {avg_train_loss:.4f}")

In [ ]:
model.eval()
val_loss = 0.0

with torch.no_grad():
    for images, masks in val_loader:
        images = images.to(device)
        masks = masks.to(device)

        outputs = model(images)
        loss = criterion(outputs, masks)
        val_loss += loss.item()

avg_val_loss = val_loss / len(val_loader)
print(f"Validation Loss: {avg_val_loss:.4f}")

In [ ]:
import matplotlib.pyplot as plt

model.eval()
images, masks = next(iter(val_loader))
images = images.to(device)

with torch.no_grad():
    outputs = model(images)
    preds = torch.argmax(outputs, dim=1)

image = images[0].cpu()
true_mask = masks[0].cpu()
pred_mask = preds[0].cpu()

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.imshow(image[0], cmap="gray")
plt.title("Input (channel 0)")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(true_mask, cmap="jet", vmin=0, vmax=3)
plt.title("True Mask")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(pred_mask, cmap="jet", vmin=0, vmax=3)
plt.title("Predicted Mask")
plt.axis("off")

plt.show()